# Track 2 Advanced (improved): both rules inside the circuit, 20 stocks

**Qiskit Fall Fest 2026, IIIT-Delhi, Track 2 · Team Wave Riders** · companion to our main Advanced notebook

**Problem.** Pick exactly K = 10 of n = 20 stocks, at most 2 per sector (5 sectors of 4), minimizing
risk − return, $C(x) = q\,x^\top\Sigma x - \mu^\top x$ with $q = 0.5$ and the README data (seed 2026).

**What changed after feedback.** Because $K = 2 \times$ (number of sectors), "at most 2 per sector" means "exactly 2
per sector". So the circuit keeps the sector rule exactly like the budget: a product of small Dicke states
$|D(4,2)\rangle$ (one per sector) as the start and XY swaps only inside each sector. No penalty, no λ, and the search
space is the $6^5 = 7{,}776$ rule-obeying portfolios.

| Section | What it shows |
|---|---|
| 1 | Problem and exact answer (brute force) |
| 2 | The sector-preserving circuit, verified against the simulator |
| 3 | Training at 20 stocks and the answer |
| 4 | Fair verdict: 8 datasets each at 12, 16, 20 stocks |
| 5 | Idea 1: parameter transfer (no training on new instances) |
| 6 | Idea 2: fewer cost terms (lossless merge + pruning) |
| 7 | Classical baselines on the same problem |
| 8 | Summary |
| 9 | Free hardware preview (FakeFez noise model) |
| 10 | **Real IBM QPU run** (about 1 minute of QPU time) |

Runs top to bottom on Colab in about 8 minutes before the QPU cell. All seeds are fixed.

## 0. Setup

In [1]:
import importlib.util, subprocess, sys
_need = {"qiskit": "qiskit==2.5.2", "qiskit_aer": "qiskit-aer==0.17.2", "qiskit_ibm_runtime": "qiskit-ibm-runtime==0.50.0",
         "scipy": "scipy", "pandas": "pandas", "matplotlib": "matplotlib"}
_missing = [pkg for mod, pkg in _need.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print("installing:", _missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)

import time, json, getpass, warnings
from itertools import combinations, product
import numpy as np, pandas as pd, scipy, qiskit, qiskit_aer, qiskit_ibm_runtime
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.quantum_info import Statevector
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke, FakeFez
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", None)
try:
    display
except NameError:                       # plain Python fallback
    display = print
SEED = 2026
print(f"python {sys.version.split()[0]} | qiskit {qiskit.__version__} | aer {qiskit_aer.__version__} | "
      f"runtime {qiskit_ibm_runtime.__version__} | numpy {np.__version__} | scipy {scipy.__version__} | pandas {pd.__version__}")

installing: ['qiskit==2.5.2', 'qiskit-aer==0.17.2', 'qiskit-ibm-runtime==0.50.0']
python 3.13.16 | qiskit 2.5.2 | aer 0.17.2 | runtime 0.50.0 | numpy 2.1.3 | scipy 1.16.3 | pandas 2.2.3


In [2]:
# ---- inlined module: advanced_core.py ----
"""Core numerics for the Track 2 Advanced notebook (n stocks, pick K, sector cap)."""
from itertools import combinations
from math import comb

import numpy as np


# ---------------------------------------------------------------- problem data
def make_instance(n=20, K=10, q=0.5, seed=2026, n_sectors=4, cap=3):
    """README generator, unchanged; sectors are contiguous blocks (not drawn from rng)."""
    if n % n_sectors:
        raise ValueError("n must be divisible by n_sectors (contiguous equal-size sectors)")
    rng = np.random.default_rng(seed)
    mu = rng.uniform(0.05, 0.25, n)
    A = rng.normal(0, 0.1, (n, n))
    sigma = A @ A.T
    sector = np.arange(n) // (n // n_sectors)
    return dict(n=n, K=K, q=q, mu=mu, sigma=sigma, sector=sector,
                n_sectors=n_sectors, cap=cap)


def feasible_states(n, K):
    """Sorted integer encodings (bit i = stock i) of every weight-K bitstring."""
    combos = np.array(list(combinations(range(n), K)), dtype=np.int64)
    ints = (np.int64(1) << combos).sum(axis=1)
    ints.sort()
    return ints


def bits_matrix(ints, n):
    return ((ints[:, None] >> np.arange(n)) & 1).astype(np.int8)


# ---------------------------------------------------------------- cost pieces
def base_cost(X, inst):
    """q x^T Sigma x - mu^T x for each row of the 0/1 matrix X."""
    Xf = X.astype(float)
    return inst["q"] * np.einsum("bi,ij,bj->b", Xf, inst["sigma"], Xf) - Xf @ inst["mu"]


def sector_excess(X, inst):
    """sum_s max(0, n_s - cap)^2 for each row of X."""
    out = np.zeros(X.shape[0])
    for s in range(inst["n_sectors"]):
        c = X[:, inst["sector"] == s].sum(axis=1)
        out += np.maximum(0, c - inst["cap"]) ** 2
    return out


def choose_lambda_s(base, excess, margin=1.05):
    """Penalty making every cap violation worse than the best cap-feasible portfolio.

    Returns (lam_s, binds). binds is False when the unconstrained optimum already obeys the cap.
    """
    gap = base[excess == 0].min() - base.min()
    if gap <= 0:
        return 0.0, False
    return margin * gap + 1e-9, True


def lambda_data_bound(inst, margin=1.05):
    """Data-only penalty strength: bounds the base-cost change of any single swap (i out, j in).

    For x' = x - e_i + e_j the exact change is
    q*(s_jj - s_ii + 2*sum_{k in x, k != i} (s_jk - s_ik)) + mu_i - mu_j; the sum runs over the
    K-1 other held stocks (the k = i cross term is already folded into s_jj - s_ii), so it is
    bounded by the K-1 largest |s_jk - s_ik| over k != i, j. The returned value is margin times
    the maximum of this bound over all (i, j), hence >= max |swap delta| over every feasible x.

    Any cap violator can be repaired by one swap whose penalty saving is >= lambda_s (the excess
    drops by at least 1) while the base cost changes by at most `max dc`, so lambda_s above this
    bound makes every violator beatable. Needs K <= cap * n_sectors (a feasible portfolio exists).
    Uses no brute-force information.
    """
    n, K, q = inst["n"], inst["K"], inst["q"]
    sigma, mu = inst["sigma"], inst["mu"]
    if K > inst["cap"] * inst["n_sectors"]:
        raise ValueError("K exceeds cap * n_sectors: no cap-feasible portfolio exists")
    worst = 0.0
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            d = np.abs(sigma[j] - sigma[i])
            d[[i, j]] = 0.0
            lin = 2.0 * np.sort(d)[-(K - 1):].sum() if K > 1 else 0.0
            quad = abs(sigma[j, j] - sigma[i, i])
            worst = max(worst, q * (lin + quad) + abs(mu[j] - mu[i]))
    return float(margin * worst)


# ---------------------------------------------------------------- Pauli-Z form
def _sector_poly_coeffs(m, cap):
    """c_k with max(0, w-cap)^2 = sum_k c_k e_k(z), z in {+1,-1}^m, w = number of -1s."""
    M = np.zeros((m + 1, m + 1))
    f = np.zeros(m + 1)
    for w in range(m + 1):
        f[w] = max(0, w - cap) ** 2
        for k in range(m + 1):
            M[w, k] = sum((-1) ** j * comb(w, j) * comb(m - w, k - j) for j in range(k + 1))
    return np.linalg.solve(M, f)


def z_terms(inst, lam_s):
    """Cost as {tuple(sorted qubits): coefficient} meaning coef * prod Z_i (empty tuple = constant)."""
    n, q = inst["n"], inst["q"]
    sigma, mu = inst["sigma"], inst["mu"]
    t = {}

    def add(key, v):
        t[key] = t.get(key, 0.0) + v

    # x_i = (1 - Z_i)/2 ;  x_i x_j = (1 - Z_i - Z_j + Z_i Z_j)/4
    for i in range(n):
        d = q * sigma[i, i] - mu[i]
        add((), d / 2)
        add((i,), -d / 2)
        for j in range(i + 1, n):
            v = q * 2 * sigma[i, j]
            add((), v / 4)
            add((i,), -v / 4)
            add((j,), -v / 4)
            add((i, j), v / 4)
    if lam_s:
        for s in range(inst["n_sectors"]):
            qs = [int(i) for i in np.where(inst["sector"] == s)[0]]
            c = _sector_poly_coeffs(len(qs), inst["cap"])
            for k in range(len(qs) + 1):
                for S in combinations(qs, k):
                    add(S, lam_s * c[k])
    return t


def eval_z_terms(terms, X):
    """Evaluate a z_terms dict on 0/1 matrix X (z_i = 1 - 2 x_i)."""
    Z = 1 - 2 * X.astype(float)
    out = np.zeros(X.shape[0])
    for S, c in terms.items():
        out += c * (np.prod(Z[:, list(S)], axis=1) if S else 1.0)
    return out


# ---------------------------------------------------------------- subspace QAOA
def ring_edges(n):
    """XY ring edges: even layer then odd layer (each layer's gates commute)."""
    if n % 2:
        raise ValueError("ring_edges needs an even n so each layer is a perfect matching")
    even = [(i, i + 1) for i in range(0, n - 1, 2)]
    odd = [(i, (i + 1) % n) for i in range(1, n, 2)]
    return even + odd


def _edge_pairs(ints, i, j):
    a = np.nonzero(((ints >> i) & 1 == 1) & ((ints >> j) & 1 == 0))[0]
    b = np.searchsorted(ints, ints[a] ^ ((1 << i) | (1 << j)))
    return a, b


class SubspaceQAOA:
    """Exact QAOA in the weight-K subspace: Dicke start, diagonal cost, XY-ring mixer."""

    def __init__(self, ints, n, cost):
        self.ints, self.n, self.cost = ints, n, cost
        self.D = len(ints)
        self.pairs = [_edge_pairs(ints, i, j) for i, j in ring_edges(n)]

    def state(self, gammas, betas):
        psi = np.full(self.D, 1 / np.sqrt(self.D), dtype=complex)
        for g, b in zip(gammas, betas):
            psi *= np.exp(-1j * g * self.cost)
            c, s = np.cos(b), -1j * np.sin(b)
            for a_idx, b_idx in self.pairs:
                pa, pb = psi[a_idx], psi[b_idx]
                psi[a_idx] = c * pa + s * pb
                psi[b_idx] = s * pa + c * pb
        return psi

    def probs(self, gammas, betas):
        return np.abs(self.state(gammas, betas)) ** 2

    def expectation(self, params):
        p = len(params) // 2
        return float(self.probs(params[:p], params[p:]) @ self.cost)

    # ---- adjoint differentiation -------------------------------------------------
    def _edge(self, psi, k, b):
        """Apply exp(-i b (XX+YY)/2) of ring edge k in place (b may be negative to undo)."""
        a_idx, b_idx = self.pairs[k]
        c, s = np.cos(b), -1j * np.sin(b)
        pa, pb = psi[a_idx], psi[b_idx]
        psi[a_idx] = c * pa + s * pb
        psi[b_idx] = s * pa + c * pb

    def _backward(self, params, psi, omega):
        """Gradient of <psi_final|diag(omega)|psi_final> by one reverse sweep (adjoint method)."""
        p = len(params) // 2
        gam, bet = params[:p], params[p:]
        psi = psi.copy()
        lam = omega * psi
        grad = np.zeros(2 * p)
        for layer in range(p - 1, -1, -1):
            gb = 0.0
            for k in range(len(self.pairs) - 1, -1, -1):
                a_idx, b_idx = self.pairs[k]
                # d/db of exp(-i b G) is -i G U; G swaps the two paired amplitudes
                gb += 2 * np.real(np.vdot(lam[a_idx], -1j * psi[b_idx])
                                  + np.vdot(lam[b_idx], -1j * psi[a_idx]))
                self._edge(psi, k, -bet[layer])
                self._edge(lam, k, -bet[layer])
            grad[p + layer] = gb
            grad[layer] = 2 * np.real(np.vdot(lam, -1j * self.cost * psi))
            ph = np.exp(1j * gam[layer] * self.cost)
            psi *= ph
            lam *= ph
        return grad

    def value_and_grad(self, params, omega):
        """(sum_i omega_i p_i, d/dparams) for a fixed real diagonal observable omega."""
        params = np.asarray(params, dtype=float)
        p = len(params) // 2
        psi = self.state(params[:p], params[p:])
        val = float(np.abs(psi) ** 2 @ omega)
        return val, self._backward(params, psi, omega)

    def cvar_value_and_grad(self, params, alpha):
        """CVaR_alpha and its gradient via Rockafellar-Uryasev with the envelope theorem."""
        params = np.asarray(params, dtype=float)
        p = len(params) // 2
        psi = self.state(params[:p], params[p:])
        pr = np.abs(psi) ** 2
        order = np.argsort(self.cost)
        cum = np.cumsum(pr[order])
        k = min(int(np.searchsorted(cum, alpha)), len(cum) - 1)
        t = self.cost[order[k]]
        omega = np.where(self.cost < t, (self.cost - t) / alpha, 0.0)
        return float(t + omega @ pr), self._backward(params, psi, omega)


def cvar(probs, cost, alpha):
    """Mean cost of the best alpha-fraction of probability mass."""
    order = np.argsort(cost)
    cum = np.cumsum(probs[order])
    k = min(int(np.searchsorted(cum, alpha)), len(cum) - 1)
    w = probs[order].copy()
    w[k + 1:] = 0
    w[k] -= cum[k] - alpha
    return float(w @ cost[order] / alpha)

In [3]:
# ---- inlined module: advanced_circuits.py ----
"""Qiskit circuits for the Advanced notebook."""
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import PauliEvolutionGate, RYGate
from qiskit.quantum_info import SparsePauliOp


def _cry(qc, theta, c, t):
    """Controlled-RY(theta) as RY(theta/2) CX RY(-theta/2) CX (two CX)."""
    qc.ry(theta / 2, t)
    qc.cx(c, t)
    qc.ry(-theta / 2, t)
    qc.cx(c, t)


def _ccry(qc, theta, c1, c2, t):
    """Doubly-controlled RY(theta): CRY(th/2)(c2,t) CX(c1,c2) CRY(-th/2)(c2,t) CX(c1,c2) CRY(th/2)(c1,t)."""
    _cry(qc, theta / 2, c2, t)
    qc.cx(c1, c2)
    _cry(qc, -theta / 2, c2, t)
    qc.cx(c1, c2)
    _cry(qc, theta / 2, c1, t)


def _scs(qc, m, k, explicit=True):
    q, p = m - 1, m - 2
    qc.cx(p, q)
    th = 2 * np.arccos(np.sqrt(1 / m))
    if explicit:
        _cry(qc, th, q, p)
    else:
        qc.append(RYGate(th).control(1, annotated=False), [q, p])
    qc.cx(p, q)
    for l in range(2, k + 1):
        t, c = m - 1 - l, m - l
        qc.cx(t, q)
        th = 2 * np.arccos(np.sqrt(l / m))
        if explicit:
            _ccry(qc, th, q, c, t)
        else:
            qc.append(RYGate(th).control(2, annotated=False), [q, c, t])
        qc.cx(t, q)


def dicke_circuit(n, k, explicit=True):
    """Baertschi-Eidenbenz Dicke-state preparation (bit i = qubit i).

    explicit=True uses hand-written CRY / CCRY decompositions; explicit=False the Qiskit
    RYGate.control(...) construction (kept for comparison; same output state).
    """
    qc = QuantumCircuit(n)
    for i in range(n - k, n):
        qc.x(i)
    for m in range(n, k, -1):
        _scs(qc, m, k, explicit)
    for m in range(k, 1, -1):
        _scs(qc, m, m - 1, explicit)
    return qc


def cost_operator(terms, n):
    """SparsePauliOp from a z_terms dict (qubit i = bit i; Qiskit labels are little-endian)."""
    labels, coefs = [], []
    for S, c in terms.items():
        lab = ["I"] * n
        for i in S:
            lab[n - 1 - i] = "Z"
        labels.append("".join(lab))
        coefs.append(c)
    return SparsePauliOp(labels, coefs).simplify()


def qaoa_circuit(n, K, H, ring, gammas, betas):
    """Dicke start + p x (exp(-i g H), XY ring mixer). exp(-i b (XX+YY)/2) = RXX(b) RYY(b)."""
    if len(gammas) != len(betas):
        raise ValueError("gammas and betas must have the same length")
    qc = dicke_circuit(n, K)
    for g, b in zip(gammas, betas):
        qc.append(PauliEvolutionGate(H, time=g), range(n))
        for i, j in ring:
            qc.rxx(b, i, j)
            qc.ryy(b, i, j)
    return qc

In [4]:
# ---- inlined module: advanced_analysis.py ----
"""Training, metrics and classical baselines for the Advanced notebook."""
import numpy as np
from scipy.optimize import minimize



# ---------------------------------------------------------------- metrics
def summarize(probs, cost, feasible_mask, optimum_idx):
    """Metrics for one distribution over the budget subspace. `feasible_mask` = obeys sector cap.

    top10 / top100 are the probability mass on the 10 / 100 lowest-cost cap-feasible portfolios.
    p_best_cond is P(best) conditioned on landing in a cap-feasible portfolio (post-selection).
    """
    p_best = float(probs[optimum_idx])
    exp = float(probs @ cost)
    feas_idx = np.where(feasible_mask)[0]
    ranked = feas_idx[np.argsort(cost[feas_idx], kind="stable")]
    p_feas = float(probs[feasible_mask].sum())
    return dict(
        p_best=p_best,
        p_best_cond=(p_best / p_feas) if p_feas > 0 else float("nan"),
        top10=float(probs[ranked[:10]].sum()),
        top100=float(probs[ranked[:100]].sum()),
        r=float((cost.max() - exp) / (cost.max() - cost.min())),
        feasible=p_feas,
        shots_99=(np.inf if p_best <= 0 else 1.0 if p_best >= 1 else float(np.ceil(np.log(0.01) / np.log1p(-p_best)))),
    )


def random_baseline(cost, feasible_mask, optimum_idx, kind="budget"):
    """Uniform random pick: kind="budget" over all budget-feasible portfolios, "cap" over cap-feasible ones."""
    if kind == "budget":
        probs = np.full(len(cost), 1 / len(cost))
    elif kind == "cap":
        probs = feasible_mask.astype(float) / feasible_mask.sum()
    else:
        raise ValueError("kind must be 'budget' or 'cap'")
    return summarize(probs, cost, feasible_mask, optimum_idx)


# ---------------------------------------------------------------- training
def interp_init(x):
    """Zhou et al. INTERP: stretch a length-p angle vector to length p+1."""
    p = len(x)
    xp = np.concatenate([[0.0], x, [0.0]])
    return np.array([((i - 1) / p) * xp[i - 1] + ((p - i + 1) / p) * xp[i] for i in range(1, p + 2)])


class Objective:
    """Training objective: `obj(params) -> float`, `obj.fun_and_jac(params) -> (float, grad)`.

    The gradient is exact (adjoint method). `nfev` counts every call of either form.
    """

    def __init__(self, sim, kind="mean", alpha=0.2):
        if kind not in ("mean", "cvar"):
            raise ValueError("kind must be 'mean' or 'cvar'")
        self.sim, self.kind, self.alpha, self.nfev = sim, kind, alpha, 0

    def __call__(self, params):
        self.nfev += 1
        p = len(params) // 2
        pr = self.sim.probs(params[:p], params[p:])
        return float(pr @ self.sim.cost) if self.kind == "mean" else cvar(pr, self.sim.cost, self.alpha)

    def fun_and_jac(self, params):
        self.nfev += 1
        params = np.asarray(params, dtype=float)
        if self.kind == "mean":
            return self.sim.value_and_grad(params, self.sim.cost)
        return self.sim.cvar_value_and_grad(params, self.alpha)

    def as_pair(self):
        """(fun, jac) pair for scipy `minimize(fun, x0, jac=jac)`; each evaluates the adjoint once."""
        cache = {}

        def vg(x):
            key = np.asarray(x, dtype=float).tobytes()
            if key not in cache:
                cache.clear()
                cache[key] = self.fun_and_jac(x)
            return cache[key]
        return (lambda x: vg(x)[0]), (lambda x: vg(x)[1])


def make_objective(sim, kind="mean", alpha=0.2):
    """Callable objective (params -> float) that also exposes `.fun_and_jac` and `.as_pair()`."""
    return Objective(sim, kind, alpha)


def _p1_grid_start(f, gammas=np.linspace(-1.5, 1.5, 9), betas=np.linspace(0, np.pi / 2, 7)):
    best = min((f(np.array([g, b])), g, b) for g in gammas for b in betas)
    return np.array([best[1], best[2]])


def _info(r, nfev):
    return dict(x=r.x.copy(), fun=float(r.fun), nfev=int(nfev), nit=int(r.nit),
                success=bool(r.success), message=str(r.message))


def train_random(sim, p, rng, kind="mean", alpha=0.2, restarts=3, maxiter=40, return_info=False):
    """Best of several random starts, exact-gradient L-BFGS-B. Default returns (x, fun)."""
    obj = make_objective(sim, kind, alpha)
    best = None
    for _ in range(restarts):
        x0 = np.concatenate([rng.uniform(-1, 1, p), rng.uniform(0, np.pi / 2, p)])
        r = minimize(obj.fun_and_jac, x0, jac=True, method="L-BFGS-B", options=dict(maxiter=maxiter))
        if best is None or r.fun < best.fun:
            best = r
    if return_info:
        return _info(best, obj.nfev)           # nfev: all restarts together
    return best.x, best.fun


def train_layerwise(sim, pmax, kind="cvar", alpha=0.2, maxiter=40, return_info=False):
    """Optimize p = 1 from a coarse grid, then grow one layer at a time from INTERP-stretched angles.

    Default returns {p: (x, fun)}; with return_info {p: dict(x, fun, nfev, nit, success, message)}
    where nfev is cumulative (grid start of 63 evaluations plus every layer up to p).
    """
    obj = make_objective(sim, kind, alpha)
    x = _p1_grid_start(obj)
    out = {}
    for p in range(1, pmax + 1):
        if p > 1:
            x = np.concatenate([interp_init(x[: p - 1]), interp_init(x[p - 1:])])
        r = minimize(obj.fun_and_jac, x, jac=True, method="L-BFGS-B", options=dict(maxiter=maxiter))
        x = r.x
        out[p] = _info(r, obj.nfev) if return_info else (x.copy(), r.fun)
    return out


def grad_variance(sim, p, rng, draws=30, eps=1e-4):
    """Variance over random angles of dE/d(gamma_1) (central difference)."""
    f = make_objective(sim, "mean")
    gs = []
    for _ in range(draws):
        x = np.concatenate([rng.uniform(-np.pi, np.pi, p), rng.uniform(0, np.pi, p)])
        xp, xm = x.copy(), x.copy()
        xp[0] += eps
        xm[0] -= eps
        gs.append((f(xp) - f(xm)) / (2 * eps))
    return float(np.var(gs))


# ---------------------------------------------------------------- classical baselines
def swap_descent(cost_fn, x):
    """Steepest-improvement 1-swap local search at fixed budget. Returns (x, cost, evaluations)."""
    x = np.array(x, dtype=int)
    cur = cost_fn(x)
    evals = 1
    while True:
        best_y, best_c = None, cur - 1e-12
        for i in np.where(x == 1)[0]:
            for j in np.where(x == 0)[0]:
                y = x.copy()
                y[i], y[j] = 0, 1
                evals += 1
                c = cost_fn(y)
                if c < best_c:
                    best_y, best_c = y, c
        if best_y is None:
            return x, cur, evals
        x, cur = best_y, best_c


def greedy_swap(cost_fn, n, K):
    """Greedy build-up of K stocks, then 1-swap local search. Returns (x, cost evaluations)."""
    evals, chosen = 0, []
    for _ in range(K):
        best, bs = None, np.inf
        for i in range(n):
            if i in chosen:
                continue
            x = np.zeros(n, dtype=int)
            x[chosen + [i]] = 1
            evals += 1
            c = cost_fn(x)
            if c < bs:
                best, bs = i, c
        chosen.append(best)
    x = np.zeros(n, dtype=int)
    x[chosen] = 1
    x, _, ev = swap_descent(cost_fn, x)
    return x, evals + ev


def simulated_annealing(cost_fn, n, K, steps, rng, t0=0.3, t1=0.005):
    """Swap-move annealing at fixed budget. Returns (best x, best cost, evaluations)."""
    x = np.zeros(n, dtype=int)
    x[rng.choice(n, K, replace=False)] = 1
    cur = cost_fn(x)
    best, bc = x.copy(), cur
    for s in range(steps):
        T = t0 * (t1 / t0) ** (s / max(steps - 1, 1))
        i = rng.choice(np.where(x == 1)[0])
        j = rng.choice(np.where(x == 0)[0])
        y = x.copy()
        y[i], y[j] = 0, 1
        c = cost_fn(y)
        if c < cur or rng.random() < np.exp(-(c - cur) / T):
            x, cur = y, c
            if cur < bc:
                best, bc = x.copy(), cur
    return best, bc, steps + 1


def wilson_interval(k, n, z=1.96):
    """Wilson score interval for a binomial proportion k/n; returns (lo, hi). n=0 gives (0, 1)."""
    if n <= 0:
        return 0.0, 1.0
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return float(max(0.0, c - h)), float(min(1.0, c + h))

## 1. The 20-stock problem and its exact answer

Brute force over all 184,756 ten-stock portfolios gives the ground truth every result is checked against.

In [5]:
t0 = time.time()
n, K, S = 20, 10, 5
inst = make_instance(n=n, K=K, seed=SEED, n_sectors=S, cap=2)
ints_all = feasible_states(n, K)
X_all = bits_matrix(ints_all, n)
base_all, exc_all = base_cost(X_all, inst), sector_excess(X_all, inst)
ok_all = exc_all == 0
opt_all = int(np.flatnonzero(ok_all)[base_all[ok_all].argmin()])
unc_all = int(base_all.argmin())
OPT_STOCKS = np.where(X_all[opt_all])[0].tolist()
print(f"all bitstrings: {2 ** n:,}; exactly 10 stocks: {len(ints_all):,}; also at most 2 per sector: {int(ok_all.sum()):,}")
print(f"unconstrained best: stocks {np.where(X_all[unc_all])[0].tolist()}, per sector "
      f"{np.bincount(np.where(X_all[unc_all])[0] // 4, minlength=S).tolist()} (breaks the rule)")
print(f"exact optimum: stocks {OPT_STOCKS}, per sector {np.bincount(np.array(OPT_STOCKS) // 4, minlength=S).tolist()}, "
      f"cost {base_all[opt_all]:.5f}  ({time.time() - t0:.0f} s)")

all bitstrings: 1,048,576; exactly 10 stocks: 184,756; also at most 2 per sector: 7,776
unconstrained best: stocks [4, 5, 6, 7, 10, 11, 12, 14, 15, 18], per sector [0, 4, 2, 3, 1] (breaks the rule)
exact optimum: stocks [1, 3, 4, 6, 10, 11, 12, 15, 17, 19], per sector [2, 2, 2, 2, 2], cost -1.11174  (1 s)


## 2. The sector-preserving circuit

- **Start:** $|D(4,2)\rangle^{\otimes 5}$, an equal superposition of the 7,776 rule-obeying portfolios.
- **Cost layer:** $e^{-i\gamma H_C}$ with risk − return only (1-body and 2-body Z terms; no penalty, no 3-body terms).
- **Mixer:** $e^{-i\beta(XX+YY)/2}$ on a 4-qubit ring inside each sector, so a stock is only ever swapped for another stock
  of the same sector and every sector keeps exactly 2.
- **Lossless merge (idea 2a):** inside a sector holding exactly 2 of 4 stocks, $Z_aZ_bZ_cZ_d = +1$, so
  $Z_aZ_b = Z_cZ_d$. The 6 within-sector pair terms merge into 3: 190 → 175 pair terms with the same cost on every
  allowed portfolio. The circuits below use the merged form.

The training simulator works directly on the 7,776 allowed portfolios; the check confirms it equals the Qiskit circuit.

In [6]:
def merged_terms(inst):
    """z_terms with each within-sector pair Z_aZ_b folded onto its complement Z_cZ_d (exact on the allowed set)."""
    out = {}
    for S_, c_ in z_terms(inst, 0.0).items():
        if len(S_) == 2 and S_[0] // 4 == S_[1] // 4:
            blk = set(range(4 * (S_[0] // 4), 4 * (S_[0] // 4) + 4))
            S_ = min(S_, tuple(sorted(blk - set(S_))))
        out[S_] = out.get(S_, 0.0) + c_
    return out

def sector_problem(n, seed):
    S_ = n // 4; K_ = 2 * S_
    inst_ = make_instance(n=n, K=K_, seed=seed, n_sectors=S_, cap=2)
    sec = [list(range(4 * s, 4 * s + 4)) for s in range(S_)]
    ints = np.array(sorted(sum(1 << i for blk in ch for i in blk)
                           for ch in product(*[list(combinations(b, 2)) for b in sec])), dtype=np.int64)
    X = bits_matrix(ints, n)
    assert (sector_excess(X, inst_) == 0).all() and len(ints) == 6 ** S_
    cost = base_cost(X, inst_)
    edges = ([(b[0], b[1]) for b in sec] + [(b[2], b[3]) for b in sec] +
             [(b[1], b[2]) for b in sec] + [(b[3], b[0]) for b in sec])
    pairs = []
    for i, j in edges:
        a = np.nonzero(((ints >> i) & 1 == 1) & ((ints >> j) & 1 == 0))[0]
        b_ = np.searchsorted(ints, ints[a] ^ ((1 << i) | (1 << j)))
        assert (ints[b_] == (ints[a] ^ ((1 << i) | (1 << j)))).all()     # a swap never leaves the allowed set
        pairs.append((a, b_))
    merged = merged_terms(inst_)
    order = sorted([S2 for S2 in merged if len(S2) == 2], key=lambda S2: -abs(merged[S2]))
    scale = 1 / (cost.max() - cost.min())
    return dict(n=n, S=S_, K=K_, inst=inst_, sec=sec, ints=ints, X=X, cost=cost, edges=edges, pairs=pairs,
                merged=merged, order=order, scale=scale, cn=(cost - cost.min()) * scale, opt=int(cost.argmin()))

def kept_terms(P, k=None):
    """k=None: all merged terms (lossless); k='full': unmerged z_terms; integer k: keep the k largest pair terms."""
    if k == "full":
        return z_terms(P["inst"], 0.0)
    if k is None:
        return P["merged"]
    keep = set(P["order"][:k])
    return {S_: c_ for S_, c_ in P["merged"].items() if len(S_) != 2 or S_ in keep}

def phase_vec(P, k=None):
    if k is None or k == "full":
        return P["cn"]
    v = eval_z_terms(kept_terms(P, k), P["X"])
    return (v - v.min()) * P["scale"]                      # same gamma scale as the full cost

def sector_state(P, g, b, ph=None):
    ph = P["cn"] if ph is None else ph
    psi = np.full(len(P["ints"]), 1 / np.sqrt(len(P["ints"])), complex)
    for gg, bb in zip(g, b):
        psi = psi * np.exp(-1j * gg * ph)
        c, s_ = np.cos(bb), -1j * np.sin(bb)
        for ia, ib in P["pairs"]:
            u, v = psi[ia].copy(), psi[ib].copy()
            psi[ia], psi[ib] = c * u + s_ * v, s_ * u + c * v
    return psi

def sector_circuit(P, g, b, k=None):
    qc = QuantumCircuit(P["n"])
    for blk in P["sec"]:
        qc.compose(dicke_circuit(4, 2), qubits=blk, inplace=True)
    H = cost_operator({S_: c_ for S_, c_ in kept_terms(P, k).items() if S_}, P["n"])
    for gg, bb in zip(g, b):
        qc.append(PauliEvolutionGate(H, time=gg * P["scale"]), range(P["n"]))
        for i, j in P["edges"]:
            qc.rxx(bb, i, j)
            qc.ryy(bb, i, j)
    return qc

def _interp(x):
    p = len(x); xp = np.concatenate([[0.0], x, [0.0]])
    return np.array([((i - 1) / p) * xp[i - 1] + ((p - i + 1) / p) * xp[i] for i in range(1, p + 2)])

def sector_train(P, p_max, k=None, alpha=0.2, maxiter=40):
    """CVaR(alpha) layer-wise from a p = 1 grid with INTERP warm starts. The circuit phases may be pruned (k), but
    the objective always uses the FULL exact cost (on a QPU each sample is scored classically)."""
    ph = phase_vec(P, k)
    f = lambda th: cvar(np.abs(sector_state(P, th[:len(th) // 2], th[len(th) // 2:], ph)) ** 2, P["cn"], alpha)
    best = min((f(np.array([g, b])), g, b) for g in np.linspace(-6, 6, 25) for b in np.linspace(0, np.pi / 2, 9))
    x, out = np.array(best[1:]), {}
    for p in range(1, p_max + 1):
        if p > 1:
            x = np.concatenate([_interp(x[:p - 1]), _interp(x[p - 1:])])
        r = minimize(f, x, method="L-BFGS-B", options=dict(maxiter=maxiter))
        x = r.x
        out[p] = (x.copy(), int(r.nfev), bool(r.success))
    return out

# ---- checks
t0 = time.time()
P20 = sector_problem(20, SEED)
assert np.where(P20["X"][P20["opt"]])[0].tolist() == OPT_STOCKS, "sector-space optimum differs from brute force"
err = np.abs(eval_z_terms(P20["merged"], P20["X"]) - eval_z_terms(z_terms(P20["inst"], 0.0), P20["X"])).max()
print(f"20 stocks: optimum over the 7,776 allowed portfolios = brute-force optimum {OPT_STOCKS}")
print(f"lossless merge: {sum(len(S_) == 2 for S_ in z_terms(P20['inst'], 0.0))} -> {len(P20['order'])} pair terms, "
      f"max |merged - full| over the 7,776 portfolios = {err:.1e}")
rng_c = np.random.default_rng(0)
for n_ in (12, 16):
    P_ = sector_problem(n_, SEED)
    for k_ in (None, len(P_["order"]) // 2):
        g_, b_ = rng_c.uniform(-3, 3, 2), rng_c.uniform(0, np.pi, 2)
        sv = Statevector(transpile(sector_circuit(P_, g_, b_, k_), optimization_level=0,
                                   basis_gates=["cx", "rz", "ry", "rx", "h", "sx", "x", "rzz", "rxx", "ryy"])).data
        ov = abs(np.vdot(sector_state(P_, g_, b_, phase_vec(P_, k_)), sv[P_["ints"]])) ** 2
        inside = float(np.sum(np.abs(sv[P_["ints"]]) ** 2))
        assert ov > 1 - 1e-9 and inside > 1 - 1e-9
        print(f"{n_} stocks, {'all merged' if k_ is None else f'{k_} pruned'} pair terms, p = 2: circuit = simulator "
              f"(overlap {ov:.10f}); probability on rule-obeying portfolios {inside:.10f}")
print(f"({time.time() - t0:.0f} s)")

20 stocks: optimum over the 7,776 allowed portfolios = brute-force optimum [1, 3, 4, 6, 10, 11, 12, 15, 17, 19]
lossless merge: 190 -> 175 pair terms, max |merged - full| over the 7,776 portfolios = 8.9e-16
12 stocks, all merged pair terms, p = 2: circuit = simulator (overlap 1.0000000000); probability on rule-obeying portfolios 1.0000000000
12 stocks, 28 pruned pair terms, p = 2: circuit = simulator (overlap 1.0000000000); probability on rule-obeying portfolios 1.0000000000
16 stocks, all merged pair terms, p = 2: circuit = simulator (overlap 1.0000000000); probability on rule-obeying portfolios 1.0000000000
16 stocks, 54 pruned pair terms, p = 2: circuit = simulator (overlap 1.0000000000); probability on rule-obeying portfolios 1.0000000000
(4 s)


## 3. Training at 20 stocks and the answer

Pre-committed recipe from the main notebook: CVaR with α = 0.2, layer-wise to p = 4 (p = 5, 6 shown for reference).
Gain = P(best) ÷ P(best) of a uniform random pick among the 7,776 rule-obeying portfolios (the fair baseline).

In [7]:
t0 = time.time()
res20 = sector_train(P20, 6)
rows = []
for p, (x, nfev, ok) in res20.items():
    pr = np.abs(sector_state(P20, x[:p], x[p:])) ** 2
    rows.append(dict(p=p, p_best=pr[P20["opt"]], gain=pr[P20["opt"]] * len(P20["ints"]),
                     shots_99=int(np.ceil(np.log(0.01) / np.log1p(-pr[P20["opt"]]))), rule_obeying_share=1.0,
                     nfev=nfev, converged=ok))
df_main = pd.DataFrame(rows)
display(df_main.round(4))
x4 = res20[4][0]
pr4 = np.abs(sector_state(P20, x4[:4], x4[4:])) ** 2
draws = np.random.default_rng(SEED).choice(len(pr4), 10_000, p=pr4 / pr4.sum())
pick = int(draws[np.argmin(P20["cost"][draws])])
PICK_STOCKS = np.where(P20["X"][pick])[0].tolist()
print(f"p = 4, 10,000 shots: best sampled portfolio {PICK_STOCKS}, per sector "
      f"{np.bincount(np.array(PICK_STOCKS) // 4, minlength=5).tolist()}, exact optimum: {pick == P20['opt']}")
print("main notebook, penalty version at p = 4: 4.4x, 49% of shots obey the sector rule, 8,096 shots for 99%; "
      "random rule-obeying picks: 35,808 draws for 99%")
print(f"({time.time() - t0:.0f} s)")

,p,p_best,gain,shots_99,rule_obeying_share,nfev,converged
0,1,0.0058,45.4070,787,1.0,45,True
1,2,0.0242,188.2655,188,1.0,120,True
2,3,0.0515,400.7136,88,1.0,294,True
3,4,0.0978,760.4256,45,1.0,396,False
4,5,0.1560,1212.8877,28,1.0,429,True
5,6,0.2001,1555.6487,21,1.0,52,True


p = 4, 10,000 shots: best sampled portfolio [1, 3, 4, 6, 10, 11, 12, 15, 17, 19], per sector [2, 2, 2, 2, 2], exact optimum: True
main notebook, penalty version at p = 4: 4.4x, 49% of shots obey the sector rule, 8,096 shots for 99%; random rule-obeying picks: 35,808 draws for 99%
(37 s)


## 4. Fair verdict: 8 datasets each at 12, 16 and 20 stocks

Same recipe (CVaR 0.2, p = 4), same seeds (2026-2033) and same pass bar (median gain ≥ 10×) as the main notebook.

In [8]:
t0 = time.time()
ver, sector_runs = [], {}
for n_ in (12, 16, 20):
    for s_ in range(2026, 2034):
        P_ = P20 if (n_, s_) == (20, SEED) else sector_problem(n_, s_)
        tr_ = res20 if (n_, s_) == (20, SEED) else sector_train(P_, 4)
        x_ = tr_[4][0]
        g_ = (np.abs(sector_state(P_, x_[:4], x_[4:])) ** 2)[P_["opt"]] * len(P_["ints"])
        evals_ = 225 + sum(tr_[p][1] for p in range(1, 5))          # 225 grid points + L-BFGS-B evaluations
        sector_runs[(n_, s_)] = dict(P=P_, x=x_, gain=g_, evals=evals_)
        ver.append(dict(n=n_, seed=s_, portfolios=len(P_["ints"]), gain=g_, training_evals=evals_))
df_ver = pd.DataFrame(ver)
df_verdict = df_ver.groupby("n").agg(portfolios=("portfolios", "first"), median_gain=("gain", "median"),
                                     min_gain=("gain", "min"), max_gain=("gain", "max"),
                                     seeds_at_least_10x=("gain", lambda v: int((v >= 10).sum())),
                                     median_training_evals=("training_evals", "median"))
display(df_verdict.round(1))
largest = df_verdict.index[df_verdict.median_gain >= 10].max()
print(f"largest size solved well (median >= 10x): {largest} stocks")
print("main notebook, penalty version, same seeds: medians 22.7x / 2.4x / 0.50x; largest solved well: 12 stocks")
print(f"({time.time() - t0:.0f} s)")

,portfolios,median_gain,min_gain,max_gain,seeds_at_least_10x,median_training_evals
n,,,,,,
12,216,44.2,33.0,54.5,8,420.0
16,1296,228.3,146.1,289.1,8,1020.5
20,7776,700.4,201.8,1638.3,8,1091.0


largest size solved well (median >= 10x): 20 stocks
main notebook, penalty version, same seeds: medians 22.7x / 2.4x / 0.50x; largest solved well: 12 stocks
(78 s)


## 5. Idea 1: parameter transfer (no training on new instances)

On a QPU every training evaluation is a batch of shots, and training costs about 1,000 evaluations per instance.
Angles trained once are reused unchanged on new instances, so the QPU only samples the final circuit.

In [9]:
t0 = time.time()
src12, src20 = sector_runs[(12, SEED)], sector_runs[(20, SEED)]
def gain_with(P, x, p=4):
    pr = np.abs(sector_state(P, x[:p], x[p:])) ** 2
    return pr[P["opt"]] * len(P["ints"])
rows = []
for n_ in (16, 20):
    for s_ in range(2026, 2034):
        r_ = sector_runs[(n_, s_)]
        row = dict(n=n_, seed=s_, gain_trained=r_["gain"], gain_angles_from_12_stocks=gain_with(r_["P"], src12["x"]),
                   training_evals_saved=r_["evals"])
        if n_ == 20 and s_ != SEED:
            row["gain_angles_from_20_stocks_seed2026"] = gain_with(r_["P"], src20["x"])
        rows.append(row)
df_transfer = pd.DataFrame(rows)
display(df_transfer.round(1))
same = df_transfer["gain_angles_from_20_stocks_seed2026"].dropna()
cross = df_transfer.groupby("n")["gain_angles_from_12_stocks"].agg(["median", "min", lambda v: int((v >= 10).sum())])
cross.columns = ["median", "min", "datasets_at_least_10x"]
print("angles from 12 stocks (seed 2026), reused at 16 and 20 stocks:"); display(cross.round(1))
TRANSFER_MEDIAN, TRANSFER_MIN = same.median(), same.min()
print(f"angles from 20 stocks (seed {SEED}) reused on the other 7 twenty-stock datasets: median {same.median():.1f}x, "
      f"min {same.min():.1f}x, {int((same >= 10).sum())}/7 at least 10x, with zero training "
      f"(trained on each: median {df_ver[df_ver.n == 20].gain.median():.1f}x after ~"
      f"{df_ver[df_ver.n == 20].training_evals.median():.0f} evaluations each)  ({time.time() - t0:.0f} s)")

,n,seed,gain_trained,gain_angles_from_12_stocks,training_evals_saved,gain_angles_from_20_stocks_seed2026
0,16,2026,289.1,83.2,824,NaN
1,16,2027,153.1,33.8,1003,NaN
2,16,2028,146.1,31.7,1168,NaN
3,16,2029,261.2,76.8,974,NaN
4,16,2030,226.8,12.1,1038,NaN
5,16,2031,185.0,37.8,1336,NaN
6,16,2032,246.8,15.8,1225,NaN
7,16,2033,229.8,16.1,995,NaN
8,20,2026,760.4,52.6,1080,NaN
9,20,2027,888.0,6.2,1398,540.8


angles from 12 stocks (seed 2026), reused at 16 and 20 stocks:


,median,min,datasets_at_least_10x
n,,,
16,32.7,12.1,8
20,32.0,6.2,6


angles from 20 stocks (seed 2026) reused on the other 7 twenty-stock datasets: median 477.4x, min 165.9x, 7/7 at least 10x, with zero training (trained on each: median 700.4x after ~1091 evaluations each)  (0 s)


## 6. Idea 2: fewer cost terms in the circuit

The stock-pair terms are about 70% of the two-qubit gates. Keep only the k largest merged pair terms in the circuit;
training and the final pick still use the full exact cost (each sample is scored classically). Tested on the main
instance across k, then for k = 120 on all 8 twenty-stock datasets.

In [10]:
t0 = time.time()
rows = []
for k in (175, 120, 80, 50, 30, 15, 0):
    x_ = sector_train(P20, 4, k=k)[4][0]
    pr = np.abs(sector_state(P20, x_[:4], x_[4:], phase_vec(P20, k))) ** 2
    rows.append(dict(pair_terms_in_circuit=k, two_qubit_gates_p4_before_routing=120 + 4 * (2 * k + 40),
                     gain=pr[P20["opt"]] * len(P20["ints"]), share_of_full=pr[P20["opt"]] * len(P20["ints"]) / sector_runs[(20, SEED)]["gain"],
                     p_find_in_100_shots=1 - (1 - pr[P20["opt"]]) ** 100))
df_sparse = pd.DataFrame(rows)
display(df_sparse.round(3))
print("unmerged circuit (190 pair terms): 120 + 4 x (2 x 190 + 40) = 1,800 two-qubit gates at p = 4 before routing")

K_KEEP = 120
rows = []
for s_ in range(2026, 2034):
    r_ = sector_runs[(20, s_)]
    x_ = sector_train(r_["P"], 4, k=K_KEEP)[4][0]
    g_ = (np.abs(sector_state(r_["P"], x_[:4], x_[4:], phase_vec(r_["P"], K_KEEP))) ** 2)[r_["P"]["opt"]] * len(r_["P"]["ints"])
    rows.append(dict(seed=s_, gain_all_terms=r_["gain"], gain_120_terms=g_, share=g_ / r_["gain"]))
df_k = pd.DataFrame(rows)
display(df_k.round(2))
SPARSE_SHARE = df_k.share.median()
print(f"k = {K_KEEP} on all 8 datasets: median {df_k.gain_120_terms.median():.1f}x (all terms {df_k.gain_all_terms.median():.1f}x), "
      f"median share kept {df_k.share.median():.0%}, {int((df_k.gain_120_terms >= 10).sum())}/8 at least 10x")

fb = FakeSherbrooke()
ROUTED, UNROUTED = {}, {}
for k in ("full", None, 120, 80, 30):
    qc_ = sector_circuit(P20, [0.5], [0.3], k)
    UNROUTED[k] = transpile(qc_, basis_gates=["cx", "rz", "sx", "x"], optimization_level=3,
                            seed_transpiler=7).count_ops().get("cx", 0)
    ROUTED[k] = min(transpile(qc_, fb, optimization_level=3, seed_transpiler=s_).count_ops().get("ecr", 0)
                    for s_ in range(1, 6))                        # best of 5 transpiler seeds
    label = {"full": "190 (unmerged)", None: "175 (lossless merge)"}.get(k, str(k))
    print(f"p = 1, {label:>20} pair terms: {UNROUTED[k]:4d} two-qubit gates before routing, "
          f"{ROUTED[k]:4d} routed ECR on FakeSherbrooke (best of 5 seeds)")
print("Before routing the saving is clear; after routing on the heavy-hex chip part of it is eaten by SWAPs,")
print("so routed counts move less (and vary with the transpiler seed).")
print(f"(main notebook, penalty version, p = 1: 3,110 routed)  ({time.time() - t0:.0f} s)")

,pair_terms_in_circuit,two_qubit_gates_p4_before_routing,gain,share_of_full,p_find_in_100_shots
0,175,1680,760.830,1.001,1.000
1,120,1240,748.437,0.984,1.000
2,80,920,672.521,0.884,1.000
3,50,680,376.759,0.495,0.993
4,30,520,129.506,0.170,0.814
5,15,400,74.785,0.098,0.620
6,0,280,2.741,0.004,0.035


unmerged circuit (190 pair terms): 120 + 4 x (2 x 190 + 40) = 1,800 two-qubit gates at p = 4 before routing


,seed,gain_all_terms,gain_120_terms,share
0,2026,760.43,748.44,0.98
1,2027,888.03,953.49,1.07
2,2028,864.22,797.29,0.92
3,2029,264.56,763.00,2.88
4,2030,201.81,182.04,0.90
5,2031,544.33,679.36,1.25
6,2032,1638.30,1556.13,0.95
7,2033,640.30,1064.10,1.66


k = 120 on all 8 datasets: median 780.1x (all terms 700.4x), median share kept 103%, 8/8 at least 10x
p = 1,       190 (unmerged) pair terms:  536 two-qubit gates before routing,  910 routed ECR on FakeSherbrooke (best of 5 seeds)
p = 1, 175 (lossless merge) pair terms:  507 two-qubit gates before routing,  877 routed ECR on FakeSherbrooke (best of 5 seeds)
p = 1,                  120 pair terms:  399 two-qubit gates before routing,  932 routed ECR on FakeSherbrooke (best of 5 seeds)
p = 1,                   80 pair terms:  319 two-qubit gates before routing,  785 routed ECR on FakeSherbrooke (best of 5 seeds)
p = 1,                   30 pair terms:  219 two-qubit gates before routing,  509 routed ECR on FakeSherbrooke (best of 5 seeds)
Before routing the saving is clear; after routing on the heavy-hex chip part of it is eaten by SWAPs,
so routed counts move less (and vary with the transpiler seed).
(main notebook, penalty version, p = 1: 3,110 routed)  (124 s)


## 7. Classical baselines on the same problem

Brute force, greedy build-up with 1-swap local search, and simulated annealing (swap moves), on the 20-stock instance.
Greedy and annealing use the data-bound sector penalty so they also search the 184,756 budget portfolios.

In [11]:
t0 = time.time()
lam_cls = lambda_data_bound(inst)
cf = lambda x: float(base_cost(x[None, :], inst)[0] + lam_cls * sector_excess(x[None, :], inst)[0])
best_cost = base_all[opt_all]
xg, ev_g = greedy_swap(cf, n, K)
rows = [dict(method="brute force over the 7,776 allowed portfolios", cost_evaluations=7776, success="always"),
        dict(method="greedy + 1-swap", cost_evaluations=ev_g, success="yes" if abs(cf(xg) - best_cost) < 1e-9 else "no")]
for steps in (500, 2000):
    hits = sum(abs(simulated_annealing(cf, n, K, steps, np.random.default_rng(1000 + s))[1] - best_cost) < 1e-9 for s in range(20))
    rows.append(dict(method=f"simulated annealing, {steps} steps", cost_evaluations=steps + 1, success=f"{hits}/20 runs"))
p4 = df_main.set_index("p").loc[4]
rows.append(dict(method="QAOA (this notebook), p = 4, trained", cost_evaluations=f"{int(p4.shots_99)} shots + ~{sector_runs[(20, SEED)]['evals']} training evaluations",
                 success="99% within those shots"))
rows.append(dict(method="QAOA, p = 4, transferred angles (idea 1)", cost_evaluations=f"~{int(np.ceil(np.log(0.01) / np.log1p(-TRANSFER_MEDIAN / 7776)))} shots, no training",
                 success="99% at the median transferred gain"))
rows.append(dict(method="uniform random rule-obeying picks", cost_evaluations=int(np.ceil(np.log(0.01) / np.log1p(-1 / 7776))),
                 success="99% within those draws"))
df_classical = pd.DataFrame(rows)
display(df_classical)
print("Reading: classical methods still find the optimum with far less work at this size. Each QAOA shot is a full")
print("circuit run, and each training evaluation is itself a batch of shots on hardware.")
print(f"({time.time() - t0:.0f} s)")

,method,cost_evaluations,success
0,"brute force over the 7,776 allowed portfolios",7776,always
1,greedy + 1-swap,556,yes
2,"simulated annealing, 500 steps",501,7/20 runs
3,"simulated annealing, 2000 steps",2001,17/20 runs
4,"QAOA (this notebook), p = 4, trained",45 shots + ~1080 training evaluations,99% within those shots
5,"QAOA, p = 4, transferred angles (idea 1)","~73 shots, no training",99% at the median transferred gain
6,uniform random rule-obeying picks,35808,99% within those draws


Reading: classical methods still find the optimum with far less work at this size. Each QAOA shot is a full
circuit run, and each training evaluation is itself a batch of shots on hardware.
(7 s)


## 8. Summary

In [12]:
v = df_verdict
print("— Both rules in the circuit (Sections 2-4) —")
print(f"Every shot obeys both rules; search space 7,776 instead of 184,756. Answer at 20 stocks: {PICK_STOCKS} "
      f"(= brute-force optimum: {PICK_STOCKS == OPT_STOCKS})")
print(f"Main instance, p = 4: gain {p4.gain:.0f}x, P(best) {p4.p_best:.1%}, {int(p4.shots_99)} shots for 99% "
      f"(penalty version: 4.4x, 8,096 shots)")
print("Median gain over 8 datasets: " + ", ".join(f"{i} stocks {r.median_gain:.1f}x ({int(r.seeds_at_least_10x)}/8 pass)"
                                                 for i, r in v.iterrows()) + "  (penalty version: 22.7x / 2.4x / 0.50x)")
print("\n— Idea 1: parameter transfer —")
print(f"20-stock angles reused on 7 new 20-stock datasets, zero training: median {TRANSFER_MEDIAN:.0f}x (min {TRANSFER_MIN:.0f}x)")
print("\n— Idea 2: fewer cost terms —")
print(f"Lossless merge 190 -> 175 terms, same answers. Keeping 120 terms: {SPARSE_SHARE:.0%} of the gain (median over "
      f"8 datasets). Two-qubit gates at p = 1 before routing: {UNROUTED['full']} -> {UNROUTED[None]} (merge) -> "
      f"{UNROUTED[120]} (120 terms); routed (best of 5): {ROUTED['full']} -> {ROUTED[None]} -> {ROUTED[120]}")
print("\n— Classical —")
print(f"Greedy + 1-swap: {ev_g} evaluations; brute force over 7,776 portfolios takes milliseconds")

— Both rules in the circuit (Sections 2-4) —
Every shot obeys both rules; search space 7,776 instead of 184,756. Answer at 20 stocks: [1, 3, 4, 6, 10, 11, 12, 15, 17, 19] (= brute-force optimum: True)
Main instance, p = 4: gain 760x, P(best) 9.8%, 45 shots for 99% (penalty version: 4.4x, 8,096 shots)
Median gain over 8 datasets: 12 stocks 44.2x (8/8 pass), 16 stocks 228.3x (8/8 pass), 20 stocks 700.4x (8/8 pass)  (penalty version: 22.7x / 2.4x / 0.50x)

— Idea 1: parameter transfer —
20-stock angles reused on 7 new 20-stock datasets, zero training: median 477x (min 166x)

— Idea 2: fewer cost terms —
Lossless merge 190 -> 175 terms, same answers. Keeping 120 terms: 103% of the gain (median over 8 datasets). Two-qubit gates at p = 1 before routing: 536 -> 507 (merge) -> 399 (120 terms); routed (best of 5): 910 -> 877 -> 932

— Classical —
Greedy + 1-swap: 556 evaluations; brute force over 7,776 portfolios takes milliseconds


## 9. Hardware: circuits and a free noisy preview

Small instances that fit today's hardware: 8 stocks (2 sectors, 36 allowed portfolios, p = 1) and 12 stocks
(3 sectors, 216 allowed portfolios, p = 1 and p = 2). Circuits use the lossless merged cost. The preview samples them
with **FakeFez**, a local copy of ibm_fez's layout and noise (free). A shot is "valid" only if it obeys both rules.

In [13]:
def build_hw_cases():
    cases = []
    for n_, p_ in ((8, 1), (12, 1), (12, 2)):
        P_ = sector_problem(n_, SEED)
        x_ = sector_train(P_, p_)[p_][0]
        cases.append(dict(label=f"S{n_}: {n_} stocks, p={p_}", P=P_, p=p_, qc=sector_circuit(P_, x_[:p_], x_[p_:]),
                          ideal=np.abs(sector_state(P_, x_[:p_], x_[p_:])) ** 2))
    return cases

def transpile_best(cases, dev):
    two_q = next(g for g in ("cz", "ecr", "cx") if g in dev.target.operation_names)
    err = float(np.median([v.error for v in dev.target[two_q].values() if v is not None and v.error is not None]))
    print(f"device: {dev.name}, native 2-qubit gate {two_q}, median error {err:.4f}")
    for c in cases:
        circ_ = c["qc"].copy(); circ_.measure_all()
        best = None
        for s_ in range(1, 6):
            tc = generate_preset_pass_manager(optimization_level=3, backend=dev, seed_transpiler=s_).run(circ_)
            g2 = int(tc.count_ops().get(two_q, 0))
            if best is None or g2 < best[1]:
                best = (tc, g2)
        c["isa"], c["n_2q"] = best
        print(f"  {c['label']}: {best[1]} {two_q} gates (ESTIMATED success {(1 - err) ** best[1]:.2e})")

def analyze(cases, counts, run_label):
    rows = []
    for c in cases:
        P_, ideal = c["P"], c["ideal"]
        allowed = set(int(v) for v in P_["ints"])
        top10 = set(int(v) for v in P_["ints"][np.argsort(P_["cost"])[:10]])
        opt_int, n_all = int(P_["ints"][P_["opt"]]), len(P_["ints"])
        noise_v = n_all / 2 ** P_["n"]
        rows.append(dict(circuit=c["label"], run="noiseless (exact)", valid_rate=1.0,
                         top10_rate_of_valid=float(ideal[np.argsort(P_["cost"])[:10]].sum()), optimum_rate_of_valid=float(ideal[P_["opt"]])))
        rows.append(dict(circuit=c["label"], run="pure noise (uniform bitstrings)", valid_rate=noise_v,
                         top10_rate_of_valid=10 / n_all, optimum_rate_of_valid=1 / n_all))
        for (lab, mode), cnt in counts.items():
            if lab != c["label"]:
                continue
            tot = sum(cnt.values())
            valid = {int(k, 2): v for k, v in cnt.items() if int(k, 2) in allowed}
            nv = sum(valid.values()); nt = sum(v for k, v in valid.items() if k in top10); no = valid.get(opt_int, 0)
            lo_v, hi_v = wilson_interval(nv, tot)
            lo_t, hi_t = wilson_interval(nt, nv) if nv else (np.nan, np.nan)
            rows.append(dict(circuit=c["label"], run=f"{run_label}, {mode}", shots=tot, valid_rate=nv / tot,
                             valid_ci=f"[{lo_v:.4f}, {hi_v:.4f}]", top10_rate_of_valid=nt / nv if nv else np.nan,
                             top10_ci=f"[{lo_t:.4f}, {hi_t:.4f}]", optimum_rate_of_valid=no / nv if nv else np.nan,
                             valid_shots=nv, optimum_shots=no))
            vv = "ABOVE" if lo_v > noise_v else ("BELOW" if hi_v < noise_v else "same as")
            vt = "ABOVE" if lo_t > 10 / n_all else ("BELOW" if hi_t < 10 / n_all else "same as")
            print(f"{c['label']}, {mode}: obey both rules {nv / tot:.1%} (CI {lo_v:.1%}-{hi_v:.1%}) {vv} pure noise "
                  f"{noise_v:.1%}; top-10 among them {nt / max(nv, 1):.1%} (CI {lo_t:.1%}-{hi_t:.1%}) {vt} random "
                  f"{10 / n_all:.1%}; optimum {no} of {nv}")
    return pd.DataFrame(rows)

t0 = time.time()
hw_cases = build_hw_cases()
fake_dev = FakeFez()
transpile_best(hw_cases, fake_dev)
preview_counts = {}
for c in hw_cases:
    sim_f = AerSimulator.from_backend(fake_dev, method="density_matrix", seed_simulator=11)
    preview_counts[(c["label"], "noisy sim")] = sim_f.run(c["isa"], shots=4096).result().get_counts()
df_preview = analyze(hw_cases, preview_counts, "FakeFez noisy simulation")
display(df_preview)
print("main notebook, penalty design on the real ibm_fez: 12 stocks (1,361 cz) = pure noise; 8 stocks with DD: "
      "30.4% valid vs 27.3% noise")
print(f"({time.time() - t0:.0f} s)")

device: fake_fez, native 2-qubit gate cz, median error 0.0039
  S8: 8 stocks, p=1: 173 cz gates (ESTIMATED success 5.08e-01)
  S12: 12 stocks, p=1: 357 cz gates (ESTIMATED success 2.48e-01)
  S12: 12 stocks, p=2: 622 cz gates (ESTIMATED success 8.78e-02)
S8: 8 stocks, p=1, noisy sim: obey both rules 60.7% (CI 59.2%-62.2%) ABOVE pure noise 14.1%; top-10 among them 78.2% (CI 76.5%-79.8%) ABOVE random 27.8%; optimum 399 of 2487
S12: 12 stocks, p=1, noisy sim: obey both rules 36.4% (CI 35.0%-37.9%) ABOVE pure noise 5.3%; top-10 among them 23.7% (CI 21.6%-26.0%) ABOVE random 4.6%; optimum 55 of 1492
S12: 12 stocks, p=2, noisy sim: obey both rules 18.2% (CI 17.0%-19.4%) ABOVE pure noise 5.3%; top-10 among them 30.3% (CI 27.1%-33.7%) ABOVE random 4.6%; optimum 72 of 745


,circuit,run,valid_rate,top10_rate_of_valid,optimum_rate_of_valid,shots,valid_ci,top10_ci,valid_shots,optimum_shots
0,"S8: 8 stocks, p=1",noiseless (exact),1.000000,0.847692,0.163200,NaN,NaN,NaN,NaN,NaN
1,"S8: 8 stocks, p=1",pure noise (uniform bitstrings),0.140625,0.277778,0.027778,NaN,NaN,NaN,NaN,NaN
2,"S8: 8 stocks, p=1","FakeFez noisy simulation, noisy sim",0.607178,0.782067,0.160434,4096.0,"[0.5921, 0.6220]","[0.7654, 0.7979]",2487.0,399.0
3,"S12: 12 stocks, p=1",noiseless (exact),1.000000,0.306181,0.054574,NaN,NaN,NaN,NaN,NaN
4,"S12: 12 stocks, p=1",pure noise (uniform bitstrings),0.052734,0.046296,0.004630,NaN,NaN,NaN,NaN,NaN
5,"S12: 12 stocks, p=1","FakeFez noisy simulation, noisy sim",0.364258,0.237265,0.036863,4096.0,"[0.3497, 0.3791]","[0.2164, 0.2595]",1492.0,55.0
6,"S12: 12 stocks, p=2",noiseless (exact),1.000000,0.519100,0.172678,NaN,NaN,NaN,NaN,NaN
7,"S12: 12 stocks, p=2",pure noise (uniform bitstrings),0.052734,0.046296,0.004630,NaN,NaN,NaN,NaN,NaN
8,"S12: 12 stocks, p=2","FakeFez noisy simulation, noisy sim",0.181885,0.303356,0.096644,4096.0,"[0.1704, 0.1940]","[0.2714, 0.3373]",745.0,72.0


main notebook, penalty design on the real ibm_fez: 12 stocks (1,361 cz) = pure noise; 8 stocks with DD: 30.4% valid vs 27.3% noise
(338 s)


## 10. Real IBM QPU run (last cell, about 1 minute of QPU time)

Submits the Section 9 circuits to a real device (ibm_fez preferred) as two jobs: without and with XY4 dynamical
decoupling, 4,096 shots per circuit. The API key and CRN are typed at run time and never stored. After the run,
paste the printed job ids into `REUSE_JOBS_HW` to reload the results later without using QPU time.

In [14]:
RUN_ON_HARDWARE = False             # False: skip this cell
REUSE_JOBS_HW = {}                 # empty = submit new jobs; after a run, paste the job ids THIS cell printed
PREFERRED_BACKEND = "ibm_fez"      # same device as the main notebook's hardware run
HW_SHOTS = 4096
OLD_PENALTY_JOBS = {"db8gl60r1c5c738mc1ug", "db8gl68r1c5c738mc1v0"}   # main notebook Section 7: different circuits

if not RUN_ON_HARDWARE:
    print("hardware run skipped (RUN_ON_HARDWARE = False)")
else:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
    if set(REUSE_JOBS_HW.values()) & OLD_PENALTY_JOBS:
        raise ValueError("Those job ids are the main notebook's penalty-design jobs. Use {} or this cell's own ids.")
    try:
        service
    except NameError:
        _key = getpass.getpass("IBM Quantum API key: ")
        _crn = getpass.getpass("Instance CRN: ")
        try:
            service = QiskitRuntimeService(channel="ibm_quantum_platform", token=_key, instance=_crn)
        except Exception:
            service = QiskitRuntimeService(channel="ibm_cloud", token=_key, instance=_crn)
    try:
        qpu = service.backend(PREFERRED_BACKEND)
    except Exception:
        qpu = service.least_busy(operational=True, simulator=False, min_num_qubits=12)
    if "hw_cases" not in globals():
        hw_cases = build_hw_cases()
    transpile_best(hw_cases, qpu)

    if REUSE_JOBS_HW:
        jobs = {m: service.job(j) for m, j in REUSE_JOBS_HW.items()}
        print("reloading saved jobs (no QPU time):", REUSE_JOBS_HW)
    else:
        jobs = {}
        for mode, dd in (("no DD", False), ("XY4 DD", True)):
            smp = SamplerV2(mode=qpu)
            smp.options.default_shots = HW_SHOTS
            if dd:
                smp.options.dynamical_decoupling.enable = True
                smp.options.dynamical_decoupling.sequence_type = "XY4"
            jobs[mode] = smp.run([c["isa"] for c in hw_cases])
            print(f"submitted {mode}: job id {jobs[mode].job_id()}")
        print("REUSE_JOBS_HW =", {m: j.job_id() for m, j in jobs.items()}, " <- paste this to reload later for free")

    hw_counts = {}
    for mode, job in jobs.items():
        print(f"waiting for '{mode}' ({job.job_id()}); queue time does not use QPU minutes ...")
        res = job.result()
        if len(res) != len(hw_cases):
            raise ValueError(f"job {job.job_id()} has {len(res)} circuits, expected {len(hw_cases)}")
        try:
            print(f"  QPU time used: {job.metrics()['usage']['quantum_seconds']} s")
        except Exception:
            pass
        for c, pub in zip(hw_cases, res):
            hw_counts[(c["label"], mode)] = pub.data.meas.get_counts()
    with open("hardware_improved_counts.json", "w") as fh:
        json.dump({"backend": qpu.name, "job_ids": {m: j.job_id() for m, j in jobs.items()},
                   "counts": {f"{lab} | {m}": v for (lab, m), v in hw_counts.items()}}, fh)
    df_hw = analyze(hw_cases, hw_counts, qpu.name)
    display(df_hw)
    print("raw counts saved to hardware_improved_counts.json")
    print("main notebook, penalty design on ibm_fez: 12 stocks = pure noise; 8 stocks with DD: valid 30.4% vs noise "
          "27.3%, top-10 18.0% vs 14.3%")

IBM Quantum API key: ··········
Instance CRN: ··········


qiskit_runtime_service._discover_account:WARNING:2026-10-15 20:06:23,132: Loading account with the given token. A saved account will not be used.


device: ibm_fez, native 2-qubit gate cz, median error 0.0029
  S8: 8 stocks, p=1: 173 cz gates (ESTIMATED success 6.00e-01)
  S12: 12 stocks, p=1: 357 cz gates (ESTIMATED success 3.49e-01)
  S12: 12 stocks, p=2: 622 cz gates (ESTIMATED success 1.60e-01)
submitted no DD: job id db8j4hgb3g5c73egd3ig
submitted XY4 DD: job id db8j4i694fic73eth2rg
REUSE_JOBS_HW = {'no DD': 'db8j4hgb3g5c73egd3ig', 'XY4 DD': 'db8j4i694fic73eth2rg'}  <- paste this to reload later for free
waiting for 'no DD' (db8j4hgb3g5c73egd3ig); queue time does not use QPU minutes ...
waiting for 'XY4 DD' (db8j4i694fic73eth2rg); queue time does not use QPU minutes ...
S8: 8 stocks, p=1, no DD: obey both rules 39.6% (CI 38.2%-41.2%) ABOVE pure noise 14.1%; top-10 among them 71.8% (CI 69.6%-73.9%) ABOVE random 27.8%; optimum 219 of 1624
S8: 8 stocks, p=1, XY4 DD: obey both rules 40.8% (CI 39.3%-42.3%) ABOVE pure noise 14.1%; top-10 among them 72.4% (CI 70.2%-74.4%) ABOVE random 27.8%; optimum 229 of 1671
S12: 12 stocks, p=1, 

,circuit,run,valid_rate,top10_rate_of_valid,optimum_rate_of_valid,shots,valid_ci,top10_ci,valid_shots,optimum_shots
0,"S8: 8 stocks, p=1",noiseless (exact),1.000000,0.847692,0.163200,NaN,NaN,NaN,NaN,NaN
1,"S8: 8 stocks, p=1",pure noise (uniform bitstrings),0.140625,0.277778,0.027778,NaN,NaN,NaN,NaN,NaN
2,"S8: 8 stocks, p=1","ibm_fez, no DD",0.396484,0.717980,0.134852,4096.0,"[0.3816, 0.4116]","[0.6956, 0.7393]",1624.0,219.0
3,"S8: 8 stocks, p=1","ibm_fez, XY4 DD",0.407959,0.723519,0.137044,4096.0,"[0.3930, 0.4231]","[0.7016, 0.7444]",1671.0,229.0
4,"S12: 12 stocks, p=1",noiseless (exact),1.000000,0.306181,0.054574,NaN,NaN,NaN,NaN,NaN
5,"S12: 12 stocks, p=1",pure noise (uniform bitstrings),0.052734,0.046296,0.004630,NaN,NaN,NaN,NaN,NaN
6,"S12: 12 stocks, p=1","ibm_fez, no DD",0.142822,0.155556,0.013675,4096.0,"[0.1324, 0.1539]","[0.1284, 0.1872]",585.0,8.0
7,"S12: 12 stocks, p=1","ibm_fez, XY4 DD",0.151855,0.180064,0.033762,4096.0,"[0.1412, 0.1632]","[0.1519, 0.2122]",622.0,21.0
8,"S12: 12 stocks, p=2",noiseless (exact),1.000000,0.519100,0.172678,NaN,NaN,NaN,NaN,NaN
9,"S12: 12 stocks, p=2",pure noise (uniform bitstrings),0.052734,0.046296,0.004630,NaN,NaN,NaN,NaN,NaN


raw counts saved to hardware_improved_counts.json
main notebook, penalty design on ibm_fez: 12 stocks = pure noise; 8 stocks with DD: valid 30.4% vs noise 27.3%, top-10 18.0% vs 14.3%
